In [2]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv
from datetime import datetime
import requests
from pymongo import errors

# Carregar variáveis de ambiente
load_dotenv("/root/sic-conti-v2/sic-backend/.env")

# Inicializar o cliente MongoDB manualmente
class Mongo:
    def __init__(self, uri):
        self.client = MongoClient(uri)
        self.db = self.client.get_database()

# Instanciar a conexão com o banco
mongo = Mongo(os.getenv("MONGO_URI"))



In [3]:
def listar_colecoes():
    return mongo.db.list_collection_names()

# Exemplo de uso da função
colecoes = listar_colecoes()
print(colecoes)


['negociacoes', 'warmup_projetos']


In [ ]:
TOKEN_RD = '65171203c468ec001837fdca'

def listar_negociacoes():
    headers = {"accept": "application/json"}
    offset = True
    page = 1
    limit = 200

    # Lista para armazenar os dados das negociações
    dados_negociacoes = []

    # Loop para obter todas as páginas da API
    while offset:
        url = f'https://crm.rdstation.com/api/v1/deals?token={TOKEN_RD}&page={page}&limit={limit}' #&win=true
        response = requests.get(url, headers=headers)

        # Verifica se a resposta foi bem-sucedida
        if response.status_code == 200:
            data = response.json()
            dados_negociacoes.extend(data['deals'])
            print(f"Página {page} - Mais negociações: {data['has_more']} - Total: {len(dados_negociacoes)}")

            # Atualiza o valor de offset e incrementa a página
            offset = data['has_more']
            page += 1
        else:
            print(f"Erro ao obter dados: {response.status_code}")
            break

    # Salvar os dados no arquivo JSON
    return dados_negociacoes
negocieacoes = listar_negociacoes()


In [22]:
len(negocieacoes)

1408

In [8]:
TOKEN_RD = '65171203c468ec001837fdca'

def listar_ganhos():
    headers = {"accept": "application/json"}
    offset = True
    page = 1
    limit = 200

    # Lista para armazenar os dados das negociações
    dados_negociacoes = []

    # Loop para obter todas as páginas da API
    while offset:
        url = f'https://crm.rdstation.com/api/v1/deals?token={TOKEN_RD}&page={page}&limit={limit}&win=true' #&win=true
        response = requests.get(url, headers=headers)

        # Verifica se a resposta foi bem-sucedida
        if response.status_code == 200:
            data = response.json()
            dados_negociacoes.extend(data['deals'])
            print(f"Página {page} - Mais negociações: {data['has_more']} - Total: {len(dados_negociacoes)}")

            # Atualiza o valor de offset e incrementa a página
            offset = data['has_more']
            page += 1
        else:
            print(f"Erro ao obter dados: {response.status_code}")
            break

    # Salvar os dados no arquivo JSON
    return dados_negociacoes


In [9]:
def iniciar_warmup(id, name, amount_total, organization):
    try:
        novo_warmup = {
            'negocio_id': id,
            'name': name,
            'organization': organization,
            'amount_total': amount_total,
            'status': 'Finalizado'
        }
        print(f'Iniciando warmup para: {novo_warmup}')

        # Obtém a coleção corretamente da instância `mongo`
        collection = mongo.db["warmup_projetos"]
        result = collection.insert_one(novo_warmup)

        print(f'Warmup iniciado com sucesso! ID do documento: {result.inserted_id}')
        return result.inserted_id  # Retorna o ID do documento inserido

    except errors.PyMongoError as e:
        print(f"Erro ao iniciar warmup: {e}")
        return None  # Retorna None em caso de erro


In [10]:
ganhos = listar_ganhos()
print(f'Ganhos obtidos: {ganhos}')
for ganho in ganhos:
    id = ganho['_id']
    name = ganho['name']
    amount_total = ganho['amount_total']
    organization = ganho['organization']['name']
    iniciar_warmup(id, name, amount_total, organization)

Página 1 - Mais negociações: True - Total: 200
Página 2 - Mais negociações: True - Total: 400
Página 3 - Mais negociações: True - Total: 600
Página 4 - Mais negociações: False - Total: 783
Ganhos obtidos: [{'_id': '6798e77cf8b9d900140e0050', 'id': '6798e77cf8b9d900140e0050', 'name': '5076.5492.2025-Hexagon _Impl Nota Serv Loader', 'amount_montly': 0.0, 'amount_unique': 5800.0, 'amount_total': 5800.0, 'prediction_date': None, 'markup': 'future', 'last_activity_at': '2025-01-28T11:36:19.994-03:00', 'interactions': 2, 'markup_last_activities': '1 day', 'created_at': '2025-01-28T11:19:40.519-03:00', 'updated_at': '2025-01-28T11:36:19.994-03:00', 'rating': 3, 'markup_created': '1 day', 'last_activity_content': None, 'user_changed': True, 'hold': None, 'win': True, 'closed_at': '2025-01-28T11:36:14.164-03:00', 'stop_time_limit': {}, 'organization': {'_id': '5ed900790b59ce00010e60f9', 'id': '5ed900790b59ce00010e60f9', 'name': '076 HEXAGON MINING TECNOLOGIA E SISTEMAS S/A', 'address': None, 'a